# Reliabilitas jawaban yes/no antar-model (Cohen's & Fleiss' Kappa)

Notebook ini mengevaluasi seberapa konsisten tiga model LLM menjawab set
pertanyaan yes/no yang sama, menggunakan hasil validasi di
`sql/requirement/question/validation/*.csv`:

- `deepseek4.1-flash-yes-no-validation.csv`
- `gemini3.1-flash-yes-no-validation.csv`
- `haiku5.5-yes-no-validation.csv`

Setiap file berisi kolom `question`, `product`, dan `answer` (format dan
delimiter berbeda-beda per file), yang akan dinormalisasi ke label biner
`1 = yes` / `0 = no` lalu digabung berdasarkan teks `question`.

Beberapa catatan kualitas data yang memengaruhi penggabungan:

- Kolom `product` **tidak** dipakai sebagai join key: `haiku5.5` memakai
  singkatan (`"UUIS"`, `"ILS"`) sedangkan `deepseek4.1-flash` dan
  `gemini3.1-flash` memakai nama penuh (`"Unified University Inventory
  System (UUIS)"`). Join dilakukan murni berdasarkan teks `question`, dan
  kolom `product` diambil dari file `deepseek4.1-flash` sebagai referensi.
- File `gemini3.1-flash` memiliki ~7 pertanyaan yang di-paraphrase
  (teksnya sedikit berbeda dari dua model lain) serta beberapa baris
  duplikat. Baris yang teksnya tidak match persis akan otomatis ter-drop
  oleh inner join, dan duplikat `question` di-dedup (ambil baris pertama)
  sebelum digabung.

Metrik yang dihitung:

1. **Cohen's Kappa** berpasangan (setiap kombinasi 2 model) - mengukur
   kesepakatan antar dua rater di luar faktor kebetulan.
2. **Fleiss' Kappa** - mengukur kesepakatan ketiga model sekaligus sebagai
   satu kelompok rater.
3. Breakdown kappa per `product`, untuk melihat apakah reliabilitas
   bervariasi antar produk/domain soal.

In [1]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

REPO_ROOT = Path.cwd().parent
VALIDATION_DIR = REPO_ROOT / "sql" / "requirement" / "question" / "validation"

MODELS = {
    "deepseek4.1-flash": {
        "file": VALIDATION_DIR / "deepseek4.1-flash-yes-no-validation.csv",
        "sep": ",",
    },
    "gemini3.1-flash": {
        "file": VALIDATION_DIR / "gemini3.1-flash-yes-no-validation.csv",
        "sep": ";",
    },
    "haiku5.5": {
        "file": VALIDATION_DIR / "haiku5.5-yes-no-validation.csv",
        "sep": ",",
    },
}

## Memuat & menormalisasi jawaban tiap model

In [2]:
def normalize_answer(value) -> int:
    """Map any yes/no spelling (0/1, yes/no, Yes/No) to a 1/0 int label."""
    text = str(value).strip().lower()
    if text in {"1", "yes", "y", "true"}:
        return 1
    if text in {"0", "no", "n", "false"}:
        return 0
    raise ValueError(f"Unrecognized answer value: {value!r}")


frames = {}
for model_name, cfg in MODELS.items():
    df = pd.read_csv(cfg["file"], sep=cfg["sep"])
    df = df[["question", "product", "answer"]].copy()
    df["answer"] = df["answer"].map(normalize_answer)
    df = df.rename(columns={"answer": model_name})

    before = len(df)
    df = df.drop_duplicates(subset="question", keep="first")
    dropped = before - len(df)
    if dropped:
        print(f"{model_name}: dropped {dropped} duplicate question row(s)")

    frames[model_name] = df
    print(f"{model_name}: {len(df)} unique rows loaded from {cfg['file'].name}")

deepseek4.1-flash: 153 unique rows loaded from deepseek4.1-flash-yes-no-validation.csv
gemini3.1-flash: dropped 2 duplicate question row(s)
gemini3.1-flash: 151 unique rows loaded from gemini3.1-flash-yes-no-validation.csv
haiku5.5: 153 unique rows loaded from haiku5.5-yes-no-validation.csv


In [3]:
model_names = list(MODELS.keys())

# Join on question text only (see note above about `product` naming
# inconsistencies); keep the `product` column from the first model.
first_model = model_names[0]
merged = frames[first_model]
for model_name in model_names[1:]:
    merged = merged.merge(
        frames[model_name].drop(columns="product"), on="question", how="inner"
    )

print(f"{len(merged)} questions matched across all {len(model_names)} models")
merged.head()

146 questions matched across all 3 models


,question,product,deepseek4.1-flash,gemini3.1-flash,haiku5.5
0,Does CCTNS require automatic fingerprint or bi...,CCTNS,0,0,0
1,Does CCTNS require a simple search to return i...,CCTNS,0,0,0
2,Does CCTNS commit to restoring the system with...,CCTNS,0,0,0
3,Is the response to an unauthorised request for...,CCTNS,1,1,1
4,May the audit trail in CCTNS be copied to remo...,CCTNS,1,1,1


## Cohen's Kappa berpasangan

Dihitung untuk setiap kombinasi 2 model, lalu diringkas dalam matriks
simetris. Skala interpretasi yang dipakai (Landis & Koch, 1977):

| Kappa | Interpretasi |
|---|---|
| < 0 | Tidak ada kesepakatan |
| 0.00 - 0.20 | Sangat rendah |
| 0.21 - 0.40 | Rendah |
| 0.41 - 0.60 | Cukup/moderat |
| 0.61 - 0.80 | Substansial |
| 0.81 - 1.00 | Hampir sempurna |

In [4]:
def interpret_kappa(k: float) -> str:
    if k < 0:
        return "tidak ada kesepakatan"
    if k <= 0.20:
        return "sangat rendah"
    if k <= 0.40:
        return "rendah"
    if k <= 0.60:
        return "cukup/moderat"
    if k <= 0.80:
        return "substansial"
    return "hampir sempurna"


pairwise_kappa = pd.DataFrame(
    np.eye(len(model_names)), index=model_names, columns=model_names
)
for a, b in itertools.combinations(model_names, 2):
    k = cohen_kappa_score(merged[a], merged[b])
    pairwise_kappa.loc[a, b] = k
    pairwise_kappa.loc[b, a] = k

pairwise_kappa.round(3)

,deepseek4.1-flash,gemini3.1-flash,haiku5.5
deepseek4.1-flash,1.000,0.973,0.986
gemini3.1-flash,0.973,1.000,0.959
haiku5.5,0.986,0.959,1.000


In [5]:
for a, b in itertools.combinations(model_names, 2):
    k = pairwise_kappa.loc[a, b]
    print(f"{a} vs {b}: kappa={k:.3f} ({interpret_kappa(k)})")

deepseek4.1-flash vs gemini3.1-flash: kappa=0.973 (hampir sempurna)
deepseek4.1-flash vs haiku5.5: kappa=0.986 (hampir sempurna)
gemini3.1-flash vs haiku5.5: kappa=0.959 (hampir sempurna)


## Fleiss' Kappa (ketiga model sekaligus)

Fleiss' Kappa mengukur reliabilitas antar-rater ketika ada lebih dari 2
rater menilai item yang sama, dengan kategori tetap (di sini: yes/no).

In [6]:
def fleiss_kappa(ratings: np.ndarray) -> float:
    """ratings: (n_items, n_categories) matrix of category counts per item."""
    n_items, n_categories = ratings.shape
    n_raters = ratings.sum(axis=1)[0]
    assert (ratings.sum(axis=1) == n_raters).all(), "Every item must have the same number of raters"

    p_j = ratings.sum(axis=0) / (n_items * n_raters)
    P_i = (np.sum(ratings * ratings, axis=1) - n_raters) / (n_raters * (n_raters - 1))
    P_bar = P_i.mean()
    P_e_bar = np.sum(p_j ** 2)

    return (P_bar - P_e_bar) / (1 - P_e_bar)


def to_rating_matrix(df: pd.DataFrame, model_cols: list[str]) -> np.ndarray:
    """Convert per-model yes/no columns into (n_items, 2) category-count matrix."""
    votes = df[model_cols].to_numpy()
    yes_counts = votes.sum(axis=1)
    no_counts = votes.shape[1] - yes_counts
    return np.column_stack([no_counts, yes_counts])


overall_matrix = to_rating_matrix(merged, model_names)
overall_fleiss = fleiss_kappa(overall_matrix)
print(f"Fleiss' Kappa (overall, {len(merged)} items, {len(model_names)} raters): "
      f"{overall_fleiss:.3f} ({interpret_kappa(overall_fleiss)})")

Fleiss' Kappa (overall, 146 items, 3 raters): 0.973 (hampir sempurna)


## Breakdown reliabilitas per product

In [7]:
rows = []
for product, group in merged.groupby("product"):
    row = {"product": product, "n_questions": len(group)}
    for a, b in itertools.combinations(model_names, 2):
        row[f"cohen[{a} vs {b}]"] = cohen_kappa_score(group[a], group[b])
    row["fleiss_kappa"] = fleiss_kappa(to_rating_matrix(group, model_names))
    rows.append(row)

per_product = pd.DataFrame(rows).set_index("product").sort_values("fleiss_kappa")
per_product.round(3)

,n_questions,cohen[deepseek4.1-flash vs gemini3.1-flash],cohen[deepseek4.1-flash vs haiku5.5],cohen[gemini3.1-flash vs haiku5.5],fleiss_kappa
product,,,,,
GAMMA-J Web Store,16,0.875,0.875,0.754,0.833
Water Use Tracking (WUT) System,14,0.857,1.000,0.857,0.904
APAF,15,1.000,1.000,1.000,1.000
CCTNS,15,1.000,1.000,1.000,1.000
Central Trading System,16,1.000,1.000,1.000,1.000
PeaZip,15,1.000,1.000,1.000,1.000
Publications Management System (PMS),11,1.000,1.000,1.000,1.000
System Administration Module of an Integrated Library System (ILS),15,1.000,1.000,1.000,1.000
Unified University Inventory System (UUIS),15,1.000,1.000,1.000,1.000


## Full agreement rate

Selain kappa, berikut proporsi pertanyaan yang dijawab sama persis oleh
ketiga model (sanity check sederhana, tanpa koreksi faktor kebetulan).

In [8]:
full_agreement = (merged[model_names].nunique(axis=1) == 1).mean()
print(f"Full 3-way agreement rate: {full_agreement:.1%} of {len(merged)} questions")

Full 3-way agreement rate: 97.9% of 146 questions
